In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.tools import tool
from typing import List

In [2]:
import datetime
from articles import ARTICLE_DB_RESULTS

In [3]:
len(ARTICLE_DB_RESULTS)

8

In [4]:
@tool
def get_top_articles_by_views(number_of_articles:int)->List:
    """Get the top articles by views. Return top `number_of_articles` articles."""
    top_articles = sorted(ARTICLE_DB_RESULTS, key=lambda x:x['views'], reverse=True)

    return top_articles[:number_of_articles]

In [5]:
@tool
def get_top_articles_by_likes(number_of_articles:int)->List:
    """Get the top articles by likes. Return top `number_of_articles` liked articles."""
    top_articles = sorted(ARTICLE_DB_RESULTS, key=lambda x:x['likes'], reverse=True)

    return top_articles[:number_of_articles]

In [6]:
@tool
def get_most_recent_articles(number_of_articles:int)->List:
    """Get the most recent articles. Return top `number_of_articles` most recent articles."""
    top_articles = sorted(ARTICLE_DB_RESULTS, key=lambda x:x['published_date'], reverse=True)

    return top_articles[:number_of_articles]

In [7]:
@tool
def get_all_articles()->List:
    """Get all articles."""
    return ARTICLE_DB_RESULTS

In [14]:
def execute_tool_calls(tool_calls:list)->list:
    result = []

    for tool_call in tool_calls:
        print(f"Executing tool call: {tool_call['name']}")
        print(f"tool args: {tool_call['args']}")

        if tool_call['name'] == 'get_top_articles_by_views':
            result.append(
                {
                    'name': tool_call['name'],
                    'result': get_top_articles_by_views.invoke(tool_call['args'])
                }
            )
        if tool_call['name'] == 'get_top_articles_by_likes':
            result.append(
                {
                    'name': tool_call['name'],
                    'result': get_top_articles_by_likes.invoke(tool_call['args'])
                }
            )
        if tool_call['name'] == 'get_most_recent_articles':
            result.append(
                {
                    'name': tool_call['name'],
                    'result': get_most_recent_articles.invoke(tool_call['args'])
                }
            )
        if tool_call['name'] == 'get_all_articles':
            result.append(
                {
                    'name': tool_call['name'],
                    'result': get_all_articles.invoke(tool_call['args'])
                }
            )
    return result


In [15]:
def initiate_the_agent(query, debug_mode=False)->List:
    """Initiate the agent with the given query."""
    load_dotenv()

    tools = [
        get_top_articles_by_views,
        get_top_articles_by_likes,
        get_most_recent_articles,
        get_all_articles
    ]

    llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(query)
    if debug_mode:
        print(response.tool_calls)
    
    return execute_tool_calls(response.tool_calls)


In [16]:
response = initiate_the_agent("Can you give me top 3 most liked and viewed articles?", True)
response

[{'name': 'get_top_articles_by_likes', 'args': {'number_of_articles': 3}, 'id': 'afoqsoja', 'type': 'tool_call'}, {'name': 'get_top_articles_by_views', 'args': {'number_of_articles': 3}, 'id': 'j86spuvf', 'type': 'tool_call'}]
Executing tool call: get_top_articles_by_likes
tool args: {'number_of_articles': 3}
Executing tool call: get_top_articles_by_views
tool args: {'number_of_articles': 3}


[{'name': 'get_top_articles_by_likes',
  'result': [{'title': 'Blockchain Beyond Cryptocurrency',
    'link': 'https://example.com/article8',
    'views': 1300,
    'likes': 400,
    'published_date': datetime.datetime(2024, 8, 20, 0, 0)},
   {'title': 'The Rise of Autonomous Vehicles',
    'link': 'https://example.com/article4',
    'views': 1200,
    'likes': 300,
    'published_date': datetime.datetime(2025, 3, 5, 0, 0)},
   {'title': 'The Future of Quantum Computing',
    'link': 'https://example.com/article2',
    'views': 1500,
    'likes': 250,
    'published_date': datetime.datetime(2025, 6, 15, 0, 0)}]},
 {'name': 'get_top_articles_by_views',
  'result': [{'title': 'The Future of Quantum Computing',
    'link': 'https://example.com/article2',
    'views': 1500,
    'likes': 250,
    'published_date': datetime.datetime(2025, 6, 15, 0, 0)},
   {'title': 'Blockchain Beyond Cryptocurrency',
    'link': 'https://example.com/article8',
    'views': 1300,
    'likes': 400,
    'publi